<a href="https://colab.research.google.com/github/syedmahmoodiagents/genai_classes/blob/main/StateManagement.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
os.environ['HF_TOKEN'] = ''

In [ ]:
llm = ChatHuggingFace(llm=HuggingFaceEndpoint(repo_id='openai/gpt-oss-20b'))

# Basic Memory using InMemoryChatMessageHistory

In [ ]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

In [ ]:
history_store = InMemoryChatMessageHistory()
history_store.add_user_message("Hello!")
history_store.add_ai_message("Hi there!")

for mess in history_store.messages:
    # print(mess.content)
    mess.pretty_print()

================================ Human Message =================================

Hello!
================================== Ai Message ==================================

Hi there!


In [ ]:
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
def get_history(session_id: str):
    return history_store # Returns the InMemoryChatMessageHistory instance

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a witty chef. Convert any request into a short recipe."),
    ("human", "How do I make a {dish}?")
])

recipe_chain = prompt | llm

chain_with_memory = RunnableWithMessageHistory(
    recipe_chain,
    get_history,
    input_messages_key="input",      # Where to find the new question
    history_messages_key="history"   # Where to inject the past context
)

config = {"configurable": {"session_id": "123"}}

res = recipe_chain.invoke({"dish": "Peanut Butter Sandwich"}, config)


/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3553: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [ ]:
res.pretty_print()

================================== Ai Message ==================================

**Peanut‑Butter Pal‑Bite**

*Ingredients*  
- 2 slices of your favorite bread (whole‑grain, sourdough, or that mystery “gluten‑free” thing)  
- 2‑3 tbsp creamy (or crunchy, if you’re feeling adventurous) peanut butter  
- A dash of sugar or honey (optional, for the sweet‑tooth brigade)  
- A pinch of sea salt (just enough to bring out the peanut’s soul)  

*Method*  
1. **Toast the bread** to your desired crisp‑ness, or leave it raw if you’re a rebel.  
2. **Spread** the peanut butter on one slice like you’re painting a masterpiece—don’t forget the edges!  
3. **Sprinkle** a whisper of sugar or a drizzle of honey for that subtle caramel kiss, and a pinch of salt for balance.  
4. **Top** with the second slice, press gently, and voilà—your sandwich is ready to be devoured!  

*Serving suggestion*  
- Slice it diagonally, serve with a glass of milk or a quick fruit smoothie, and enjoy the nutty hug that fol

# Using MessagePlaceholder

In [ ]:

from langchain_core.prompts import MessagesPlaceholder

# Upgrade the prompt to include a "history" placeholder slot
history_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful companion."),
    MessagesPlaceholder(variable_name="history"), # History injects here
    ("human", "{input}")
])

# Build the foundational chain
base_chain = history_prompt | llm

# Memory storage dictionary
session_store = {}

def get_session_history(session_id: str):
    if session_id not in session_store:
        session_store[session_id] = InMemoryChatMessageHistory()
    return session_store[session_id]

# Wrap the foundational chain to make it smart
conversational_chain = RunnableWithMessageHistory(
    base_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history"
)

# Run with config session routing
config = {"configurable": {"session_id": "001"}}


/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3553: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [ ]:
conversational_chain.invoke({"input": "what is the capital of France?"}, config)

AIMessage(content='The capital of France is **Paris**. It’s the country’s political, cultural, and economic center, home to iconic landmarks such as the Eiffel Tower, Notre-Dame Cathedral, and the Louvre Museum.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 75, 'prompt_tokens': 87, 'total_tokens': 162}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_996f667773', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a01a50-d6d5-7cd0-9aaf-f9f2b3ddbdd2-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 87, 'output_tokens': 75, 'total_tokens': 162})

In [ ]:
memresp = conversational_chain.invoke({"input": "What is spoken over there?"}, config)

In [ ]:
memresp.pretty_print()

================================== Ai Message ==================================

In France, the official language is **French**. It’s used in all official, educational, and everyday contexts.

Additionally, several regional languages and dialects are spoken in specific parts of the country, such as Breton in Brittany, Alsatian in Alsace, Occitan in the south, Corsican in Corsica, Basque in the far‑west, and others. However, French remains the lingua franca nationwide.


In [ ]:
!pip install langchain_huggingface langgraph --q

In [ ]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

In [ ]:
from langchain.agents import create_agent

In [ ]:
def get_info() -> str:
    """Look up information about the current user."""
    return "No user profile on file."


In [ ]:
agent = create_agent(
    model = llm,
    tools = [get_info],
    checkpointer = InMemorySaver()
)

In [ ]:
thread_config = {"configurable": {"thread_id": "003"}}

In [ ]:
agent.invoke(
    {"messages": [{"role":"user", "content": "My name is David"}]}, thread_config
)

{'messages': [HumanMessage(content='My name is David', additional_kwargs={}, response_metadata={}, id='6313a640-f729-458e-a38d-1358c49997a5'),
  AIMessage(content='Nice to meet you, David! How can I help you today?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 49, 'prompt_tokens': 121, 'total_tokens': 170}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_92d51d08e5', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a01865-b3bc-7622-b9f7-14002024d91a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 121, 'output_tokens': 49, 'total_tokens': 170})]}

In [ ]:
agent.invoke(
    {"messages": [{"role":"user", "content": "my freinds name is Bob"}]}, thread_config
)

{'messages': [HumanMessage(content='My name is David', additional_kwargs={}, response_metadata={}, id='6313a640-f729-458e-a38d-1358c49997a5'),
  AIMessage(content='Nice to meet you, David! How can I help you today?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 49, 'prompt_tokens': 121, 'total_tokens': 170}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_92d51d08e5', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a01865-b3bc-7622-b9f7-14002024d91a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 121, 'output_tokens': 49, 'total_tokens': 170}),
  HumanMessage(content='my freinds name is Bob', additional_kwargs={}, response_metadata={}, id='d7082f66-9d61-458b-9dfb-68752571ff70'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{}', 'name': 'get_info', 'description': None}, 'id': 'fc_bec7acc9-9c71-429b-9ab0-9041f943ebe8', 'type': 'function'}]}, response_metadata={'toke

In [ ]:
respo = agent.invoke(
    {"messages": [{"role":"user", "content": "what was I talking about"}]}, thread_config
)

In [ ]:
for mess in respo["messages"]:
    print(mess.content)

My name is David
Nice to meet you, David! How can I help you today?
my freinds name is Bob

No user profile on file.
Got it—your friend’s name is Bob. How can I help you with that? Whether you’re looking to set up a contact, plan a meetup, or something else, just let me know!
what was I talking about
You were just letting me know who you are and who your friend is.  
- You’re David.  
- Your friend’s name is Bob.  

If you’d like to keep talking about that or move on to something else, just let me know!
what was I talking about
You just let me know:

- Your name is **David**.  
- Your friend’s name is **Bob**.

That’s all I have from our conversation so far. If there’s something else you’d like to discuss or if you want to continue with a different topic, just let me know!


In [ ]:
for mess in respo["messages"]:
    print(mess.pretty_print())

================================ Human Message =================================

My name is David
None
================================== Ai Message ==================================

Nice to meet you, David! How can I help you today?
None
================================ Human Message =================================

my freinds name is Bob
None
================================== Ai Message ==================================
Tool Calls:
  get_info (fc_bec7acc9-9c71-429b-9ab0-9041f943ebe8)
 Call ID: fc_bec7acc9-9c71-429b-9ab0-9041f943ebe8
  Args:
None
================================= Tool Message =================================
Name: get_info

No user profile on file.
None
================================== Ai Message ==================================

Got it—your friend’s name is Bob. How can I help you with that? Whether you’re looking to set up a contact, plan a meetup, or something else, just let me know!
None
================================ Human Message ==================

In [ ]:
thread_config2 = {"configurable": {"thread_id": "007"}}

In [ ]:
agent2 = create_agent(
    model = llm,
    tools = [],
    checkpointer = InMemorySaver()
)

In [ ]:
agent2.invoke(
    {"messages": [{"role":"user", "content": "what is capital of France"}]}, thread_config2
)

{'messages': [HumanMessage(content='what is capital of France', additional_kwargs={}, response_metadata={}, id='9d08fc98-1027-4b73-b376-ef90f721e140'),
  AIMessage(content='The capital of France is **Paris**.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 37, 'prompt_tokens': 76, 'total_tokens': 113}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_9340e7d14d', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a01858-f8cd-7f41-9a6c-c0d0c643771d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 76, 'output_tokens': 37, 'total_tokens': 113})]}

In [ ]:
agent2.invoke(
    {"messages": [{"role":"user", "content": "what is spoken over there"}]}, thread_config2
)

{'messages': [HumanMessage(content='what is capital of France', additional_kwargs={}, response_metadata={}, id='9d08fc98-1027-4b73-b376-ef90f721e140'),
  AIMessage(content='The capital of France is **Paris**.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 37, 'prompt_tokens': 76, 'total_tokens': 113}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_9340e7d14d', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a01858-f8cd-7f41-9a6c-c0d0c643771d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 76, 'output_tokens': 37, 'total_tokens': 113}),
  HumanMessage(content='what is spoken over there', additional_kwargs={}, response_metadata={}, id='f228747e-785a-4634-846e-aaf0004136fc'),
  AIMessage(content="In France, the official language is **French**. It's the primary language used in everyday communication, government, education, media, and most public services. While French is dominant, you may also hear regiona

# StateGraph

In [ ]:
from langgraph.graph import StateGraph, START, END

In [ ]:
def add_ten(sta: dict):
    val = sta["curr"] + 10
    return {"curr": val, "history": sta["history"] + [val]}

def multiply_by_two(st: dict):
    val = st["curr"] * 2
    return {"curr": val, "history": st["history"] + [val]}


In [ ]:
builder = StateGraph(dict)

In [ ]:
builder.add_node("step_one", add_ten)
builder.add_node("step_two", multiply_by_two)

In [ ]:
builder.add_edge(START, "step_one")
builder.add_edge("step_one", "step_two")
builder.add_edge("step_two", END)

In [ ]:
pipeline = builder.compile(checkpointer=InMemorySaver())

In [ ]:
config = {"configurable": {"thread_id": "m01"}}

In [ ]:
result = pipeline.invoke({"curr": 5, "history": [5]}, config)

In [ ]:
result

{'current_value': 30, 'history': [5, 15, 30]}

In [ ]:
result = pipeline.invoke({"curr": result["curr"], "history": result["history"]}, config) # Fixed: Pass entire state including 'history'

In [ ]:
result

{'current_value': 80, 'history': [5, 15, 30, 40, 80]}

# Using State as TypeDict

In [ ]:
from typing import TypedDict

In [ ]:
class PipelineState(TypedDict):
    curr: int
    history: list[int]

def add_ten(stat: PipelineState):
    val = stat["curr"] + 10
    return {"curr": val, "history": stat["history"] + [val]}

def multiply_by_two(stte: PipelineState):
    val = stte["curr"] * 2
    return {"curr": val, "history": stte["history"] + [val]}


builder = StateGraph(PipelineState)
builder.add_node("step_one", add_ten)
builder.add_node("step_two", multiply_by_two)

builder.add_edge(START, "step_one")
builder.add_edge("step_one", "step_two")
builder.add_edge("step_two", END)


pipeline = builder.compile(checkpointer=InMemorySaver())

config = {"configurable": {"thread_id": "math_session_02"}}

result = pipeline.invoke({"curr": 5, "history": [5]}, config)
print(result)

next_result = pipeline.invoke({"curr": result["curr"]}, config)
print(next_result)

{'curr': 30, 'history': [5, 15, 30]}
{'curr': 80, 'history': [5, 15, 30, 40, 80]}
